# NILM Commissioning Manager

**Non-Intrusive Load Monitoring (NILM)** is the process of disaggregating a household's total power consumption into individual appliance-level signals — without installing per-device sensors.

This notebook implements the **commissioning (calibration) phase**: a one-time setup where the user manually tags each appliance by pressing ON and OFF on a mobile app while the system records the aggregate power draw. From this tagged session, a physical profile is extracted for each appliance and stored for use by the downstream disaggregation model.

## What this notebook does

1. **`ApplianceType`** — an enum defining the supported appliance categories
2. **`ApplianceProfile`** — a typed dataclass storing the four extracted physical characteristics per appliance:
   - `peak_power_w` — inrush power at turn-on (W)
   - `steady_state_power_w` — average operating power during normal run (W)
   - `settle_time_seconds` — time for power to drop from peak to within 5% of steady state
   - `variance_w` — variance of the steady-state draw (W²), capturing how noisy the appliance is
3. **`CommissioningManager`** — processes a tagged power recording and populates an in-memory profile database
4. **Simulation examples** — three synthetic calibration sessions (TV, refrigerator, heat pump) demonstrating how different appliance types produce distinct profiles

## Key design decisions

- The house's **baseline load is subtracted** at the moment the user taps ON, isolating only the new appliance's contribution
- Steady state is estimated from the **final 20%** of the recording window, assuming the transient has passed by then
- Settle time is measured as the number of samples after the peak until power falls to within **5% of steady state**, converted to seconds via the sampling rate

In [1]:
import numpy as np
from enum import Enum, auto
from dataclasses import dataclass
from typing import Dict, List

class ApplianceType(Enum):
    WASHING_MACHINE = auto()
    REFRIGERATOR = auto()
    HEAT_PUMP = auto()
    OVEN = auto()

@dataclass
class ApplianceProfile:
    """A strictly typed profile storing the extracted physical characteristics of the appliance."""
    appliance_type: ApplianceType
    peak_power_w: float
    steady_state_power_w: float
    settle_time_seconds: int
    variance_w: float

class CommissioningManager:
    def __init__(self, sampling_rate_hz: int = 1):
        """
        Initializes the manager responsible for processing user-tagged training data.
        """
        self.sampling_rate_hz = sampling_rate_hz
        # Simulating a database table or in-memory store for the generated profiles
        self.database: Dict[ApplianceType, ApplianceProfile] = {}
        
    def calibrate_appliance(self, appliance_type: ApplianceType, recorded_power: List[float], baseline_power: float) -> ApplianceProfile:
        """
        Processes the recorded power stream from the moment the user tapped 'ON' 
        until they tapped 'OFF', extracting the core features.
        """
        power_array = np.array(recorded_power)
        
        # 1. Isolate the appliance's actual draw by subtracting the house's baseline power
        # at the moment the user pressed 'ON'.
        isolated_power = power_array - baseline_power
        
        # 2. Extract Peak Power (Inrush)
        peak_idx = np.argmax(isolated_power)
        peak_power = float(isolated_power[peak_idx])
        
        # 3. Extract Steady-State Power
        # We assume the final 20% of the user's recorded session represents the normalized operating state.
        tail_length = max(1, int(len(isolated_power) * 0.2))
        tail_data = isolated_power[-tail_length:]
        steady_state_power = float(np.mean(tail_data))
        variance = float(np.var(tail_data))
        
        # 4. Calculate Settle Time (Transient Duration)
        # Find the first index after the peak where the power drops to within 5% of the steady state.
        settle_idx = peak_idx
        threshold = steady_state_power * 1.05
        
        for i in range(peak_idx, len(isolated_power)):
            if isolated_power[i] <= threshold:
                settle_idx = i
                break
                
        settle_time_seconds = int((settle_idx - peak_idx) / self.sampling_rate_hz)
        
        # 5. Create and store the structured profile
        profile = ApplianceProfile(
            appliance_type=appliance_type,
            peak_power_w=round(peak_power, 2),
            steady_state_power_w=round(steady_state_power, 2),
            settle_time_seconds=settle_time_seconds,
            variance_w=round(variance, 2)
        )
        
        self.database[appliance_type] = profile
        return profile

# --- Execution Example ---
if __name__ == "__main__":
    manager = CommissioningManager(sampling_rate_hz=1)
    
    # Baseline grid load before the user taps "ON"
    house_baseline = 300.0
    
    # --- SIMULATION 1: The Television ---
    # Characteristic: Almost zero inrush spike, highly stable power draw immediately.
    # Expected: Peak power should equal steady state, settle time should be 0.
    tv_session = [
        450.0, 450.0, 451.0, 450.0, 449.0, 450.0, 450.0, 450.0
    ]
    
    # --- SIMULATION 2: The Refrigerator ---
    # Characteristic: Sharp compressor spike, quick drop, stable cooling draw.
    # Expected: High peak, low settle time, low variance.
    fridge_session = [
        1200.0, 600.0, 500.0, 450.0, 450.0, 450.0, 455.0, 450.0, 450.0
    ]
    
    # --- SIMULATION 3: The Heat Pump ---
    # Characteristic: Massive inrush spike, slower settling time, fluctuating steady state.
    # Expected: Highest peak, longer settle time, higher variance.
    heat_pump_session = [
        2800.0, 2400.0, 2000.0, 1800.0, 1500.0, 1550.0, 1480.0, 1520.0, 1500.0, 1510.0
    ]

    # --- Execution and Output ---
    test_cases = [
        (ApplianceType.OVEN, tv_session, "Television (Purely Resistive/Electronic)"), # Reusing OVEN enum slot for demo, or add TELEVISION to Enum
        (ApplianceType.REFRIGERATOR, fridge_session, "Refrigerator (Inductive Compressor)"),
        (ApplianceType.HEAT_PUMP, heat_pump_session, "Heat Pump (Heavy Inductive/Variable)")
    ]

    for app_type, session_data, description in test_cases:
        print(f"\n--- Running Calibration: {description} ---")
        profile = manager.calibrate_appliance(
            appliance_type=app_type,
            recorded_power=session_data,
            baseline_power=house_baseline
        )
        
        print(f"Extracted Profile Metrics:")
        print(f"  Peak Inrush:   {profile.peak_power_w} W")
        print(f"  Steady State:  {profile.steady_state_power_w} W")
        print(f"  Settle Time:   {profile.settle_time_seconds} seconds")
        print(f"  Variance:      {profile.variance_w} W^2")


--- Running Calibration: Television (Purely Resistive/Electronic) ---
Extracted Profile Metrics:
  Peak Inrush:   151.0 W
  Steady State:  150.0 W
  Settle Time:   0 seconds
  Variance:      0.0 W^2

--- Running Calibration: Refrigerator (Inductive Compressor) ---
Extracted Profile Metrics:
  Peak Inrush:   900.0 W
  Steady State:  150.0 W
  Settle Time:   3 seconds
  Variance:      0.0 W^2

--- Running Calibration: Heat Pump (Heavy Inductive/Variable) ---
Extracted Profile Metrics:
  Peak Inrush:   2500.0 W
  Steady State:  1205.0 W
  Settle Time:   4 seconds
  Variance:      25.0 W^2
